# Visual Data Analysis of Fraudulent Transactions

Exploratory analysis of 3,500 credit-card transactions (Jan–Dec 2018, 25 card holders, 150 merchants) stored in PostgreSQL.

| # | Question | Technique |
|---|----------|-----------|
| 1 | Do large charges cluster in the early-morning hours (07:00–09:00)? | Time-window filter + scatter plot |
| 2 | How do the spending patterns of card holders **2** and **18** differ? | Summary statistics + hvPlot time series |
| 3 | Did card holder **25** have unauthorised charges in Jan–Jun 2018? | Monthly box plots + IQR flagging |

**Before running:** load `SQL/schema.sql`, `SQL/seed.sql` and `SQL/views.sql`, then set `DATABASE_URL` (see the README).

In [1]:
import calendar

import hvplot.pandas  # noqa: F401  (registers the .hvplot accessor)
import pandas as pd
import plotly.express as px
import plotly.io as pio

from db_utils import get_engine, load_transactions

# Light-weight Plotly output that renders in JupyterLab / VS Code
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

## Load the data
Everything comes from the `v_transaction_details` view, which joins transactions to card holders, merchants and categories.

In [2]:
engine = get_engine()
transactions = load_transactions(engine)

print(f"{len(transactions):,} transactions | {transactions['holder_id'].nunique()} card holders | "
      f"{transactions['merchant_id'].nunique()} merchants")
print(f"Period: {transactions.index.min():%Y-%m-%d} to {transactions.index.max():%Y-%m-%d}")
transactions.head()

3,500 transactions | 25 card holders | 150 merchants
Period: 2018-01-01 to 2018-12-31


,transaction_id,amount,card,holder_id,holder_name,merchant_id,merchant_name,category
date,,,,,,,,
2018-01-01 21:35:10,222,6.22,3561954487988605,13,John Martin,69,Dominguez PLC,food truck
2018-01-01 21:43:12,2045,3.83,5135837688671496,13,John Martin,85,Patton-Rivera,bar
2018-01-01 22:41:21,395,9.61,213193946980303,10,Matthew Gutierrez,82,Day-Murray,food truck
2018-01-01 23:13:30,3309,19.03,4263694062533017,4,Danielle Green,5,Miller-Blevins,pub
2018-01-01 23:15:10,567,2.95,4498002758300,18,Malik Carlson,64,"Cline, Myers and Strong",restaurant


## 1. Early-hour transactions (07:00–09:00)

Large purchases at unusual hours are a classic fraud signal. Below are the 100 highest-value transactions made between 07:00 and 09:00.

In [3]:
early_hours = transactions.between_time("07:00", "09:00")
top_100_early = early_hours.sort_values("amount", ascending=False).head(100)

print(f"{len(early_hours)} transactions fall in the window; the top 100 start at ${top_100_early['amount'].min():.2f}")
top_100_early.head(10)

275 transactions fall in the window; the top 100 start at $11.65


,transaction_id,amount,card,holder_id,holder_name,merchant_id,merchant_name,category
date,,,,,,,,
2018-12-07 07:22:03,3163,"1,894.00",4761049645711555811,1,Robert Johnson,9,"Curry, Scott and Richardson",bar
2018-03-05 08:26:08,2451,"1,617.00",5570600642865857,16,Crystal Clark,4,Mccarty-Thomas,bar
2018-03-06 07:18:09,2840,"1,334.00",4319653513507,25,Nancy Contreras,87,Griffin-Woodard,bar
2018-01-22 08:07:03,1442,"1,131.00",5570600642865857,16,Crystal Clark,144,"Walker, Deleon and Wolf",restaurant
2018-09-26 08:48:40,968,"1,060.00",4761049645711555811,1,Robert Johnson,134,"Jenkins, Peterson and Beck",restaurant
2018-09-06 08:28:55,1368,"1,017.00",4761049645711555811,1,Robert Johnson,135,"Jacobs, Torres and Walker",bar
2018-03-26 07:41:59,1620,"1,009.00",30181963913340,9,Laurie Gibbs,111,Padilla-Clements,coffee shop
2018-12-14 08:51:41,208,748.00,501879657465,12,Megan Price,96,Johnson-Fuller,pub
2018-04-01 07:17:21,774,100.00,4319653513507,25,Nancy Contreras,111,Padilla-Clements,coffee shop


In [4]:
px.scatter(
    top_100_early.reset_index(),
    x="date", y="amount", color="category", size="amount",
    hover_data=["holder_name", "merchant_name"],
    labels={"date": "Date", "amount": "Amount ($)", "category": "Category"},
    title="Top 100 transactions between 07:00 and 09:00",
)

In [5]:
# Where do the *large* early-morning charges (> $100) come from?
(early_hours[early_hours["amount"] > 100]
    .groupby("category")["amount"]
    .agg(transactions="count", total="sum", largest="max")
    .sort_values("transactions", ascending=False))

,transactions,total,largest
category,,,
bar,4,"5,862.00","1,894.00"
restaurant,2,"2,191.00","1,131.00"
coffee shop,1,"1,009.00","1,009.00"
pub,1,748.00,748.00


**Observation.** 275 of the 3,500 transactions fall in this window. The top 100 contain a sharp break: 91 of them are ordinary charges between $11.65 and $23.13, while eight charges are above $700 and reach nearly $1,900 (plus one at exactly $100). Those eight are spread across four categories — **bar** (4 transactions, $5,862 total), **restaurant** (2), **coffee shop** (1) and **pub** (1) — so no single category explains them. A $1,009 coffee-shop charge at breakfast time is the most out of place.

## 2. Card holders 2 and 18: contrasting spending patterns

Use hvPlot to compare the two holders' transaction history across 2018, then back the visual impression with numbers.

In [6]:
pair = transactions[transactions["holder_id"].isin([2, 18])]

summary = pair.groupby("holder_name")["amount"].agg(
    transactions="count",
    mean="mean",
    median="median",
    max="max",
    over_100=lambda s: (s > 100).sum(),
    under_2=lambda s: (s < 2).sum(),
)
summary["under_2_pct"] = 100 * summary["under_2"] / summary["transactions"]
summary

,transactions,mean,median,max,over_100,under_2,under_2_pct
holder_name,,,,,,,
Malik Carlson,133,90.98,10.37,"1,839.00",12,19,14.29
Shane Shaffer,99,9.59,10.26,19.51,0,11,11.11


In [7]:
holder_2 = pair[pair["holder_id"] == 2]["amount"].hvplot(
    label="Holder 2 (Shane Shaffer)", xlabel="Date", ylabel="Amount ($)")
holder_18 = pair[pair["holder_id"] == 18]["amount"].hvplot(
    label="Holder 18 (Malik Carlson)", xlabel="Date", ylabel="Amount ($)")

(holder_2 * holder_18).opts(title="Transaction amounts in 2018: holder 2 vs holder 18", width=900, height=380, legend_position="top")

:Overlay
   .Curve.Holder_2_left_parenthesis_Shane_Shaffer_right_parenthesis  :Curve   [date]   (amount)
   .Curve.Holder_18_left_parenthesis_Malik_Carlson_right_parenthesis :Curve   [date]   (amount)

### Conclusions for question 2

* **Holder 2 is consistent.** 99 transactions, none above **$19.51**, median ≈ $10. The pattern never leaves the cardholder's normal range, and only 11% of charges are under $2.
* **Holder 18 has a clear anomaly.** The median is almost identical (≈ $10), but 7 charges exceed $900 (up to **$1,839**) and the average is pulled up to ≈ $91. These large charges hit restaurants, a pub and a food truck, several at odd hours (e.g. 03:23 and 05:30).
* **Micro-transactions do not separate them.** Holder 18 has 19 charges under $2 (14.3% — the highest share of any holder) against 11 for holder 2 (11.1%).

An earlier version of this analysis called holder 2 the more suspicious account because of the "many small transactions". The data do not support that: small charges are normal for both, and it is **holder 18's large, irregular charges** that warrant a fraud review.

## 3. Card holder 25: suspected unauthorised restaurant charges (Jan–Jun 2018)

The card holder suspects someone used her corporate card for expensive restaurant bills in the first half of 2018. One box plot per month shows how many charges fall outside the usual range.

In [8]:
MONTHS = list(calendar.month_name)[1:7]  # January ... June

holder_25 = transactions.loc[
    (transactions["holder_id"] == 25) & (transactions.index < "2018-07-01")
].copy()
holder_25["month"] = holder_25.index.month_name()
holder_25["hour"] = holder_25.index.hour

print(f"{len(holder_25)} transactions for {holder_25['holder_name'].iloc[0]} between January and June 2018")

68 transactions for Nancy Contreras between January and June 2018


In [9]:
fig = px.box(
    holder_25.reset_index(), x="month", y="amount", color="category",
    category_orders={"month": MONTHS},
    hover_data=["merchant_name", "date"],
    labels={"month": "Month", "amount": "Amount ($)", "category": "Category"},
    title="Monthly transaction amounts: card holder 25 (Jan–Jun 2018)",
)
fig

In [10]:
# Flag the charges Plotly shows as outliers, using the same 1.5 x IQR rule per month
quartiles = holder_25.groupby("month")["amount"].quantile([0.25, 0.75]).unstack()
upper_fence = quartiles[0.75] + 1.5 * (quartiles[0.75] - quartiles[0.25])

flagged = holder_25[holder_25["amount"] > holder_25["month"].map(upper_fence)].reset_index()
flagged["month"] = pd.Categorical(flagged["month"], MONTHS, ordered=True)
flagged = flagged.sort_values(["month", "amount"], ascending=[True, False])

print(f"{len(flagged)} flagged charges, {(flagged['amount'] > 1000).sum()} of them above $1,000")
flagged[["month", "date", "amount", "category", "merchant_name", "hour"]]

9 flagged charges, 6 of them above $1,000


,month,date,amount,category,merchant_name,hour
0,January,2018-01-30 18:31:00,"1,177.00",restaurant,"Cline, Myers and Strong",18
1,March,2018-03-06 07:18:09,"1,334.00",bar,Griffin-Woodard,7
3,April,2018-04-08 06:03:50,"1,063.00",pub,"Bryant, Thomas and Collins",6
4,April,2018-04-09 18:28:25,269.00,restaurant,Hamilton-Mcfarland,18
2,April,2018-04-01 07:17:21,100.00,coffee shop,Padilla-Clements,7
5,May,2018-05-13 06:31:20,"1,046.00",food truck,Baker Inc,6
8,June,2018-06-22 06:16:50,"1,813.00",bar,"Cox, Montgomery and Morgan",6
6,June,2018-06-04 03:46:15,"1,162.00",pub,Johnson-Fuller,3
7,June,2018-06-06 21:50:17,749.00,restaurant,Hamilton-Mcfarland,21


### Conclusions for question 3

* Charges are flagged in **five of the six months** (none in February). Nine in total — six of them above **$1,000**: $1,813 (bar), $1,334 (bar), $1,177 (restaurant), $1,162 (pub), $1,063 (pub) and $1,046 (food truck).
* The suspicion is therefore **wider than restaurants**: the large charges are spread over bars, pubs, a restaurant and a food truck, while her normal charge is about $10–$15.
* **Timing strengthens the case:** five of those six large charges were made between **03:46 and 07:18**, hours at which ordinary meals are unusual.
* An earlier note singled out food trucks, but only one food-truck charge ($1,046) is anomalous; the rest of the flagged charges are at bars, pubs and a restaurant. The three smaller flags ($749, $269 and $100) are worth a quick check but are less clear-cut.

Recommendation: contact the card holder about the six charges over $1,000 first.

## Key takeaways

1. **Size and time matter more than count.** Micro-transaction counts track overall activity (correlation ≈ 0.88 across holders); large charges at odd hours are the better signal.
2. **Holder 18** and **holder 25** each show several $1,000+ charges that sit far outside their normal ~$10–$15 range.
3. Statistical rules are only a starting point — see `outlier_challenge.ipynb` for how 3σ and IQR compare.